Environment

In [ ]:
import sys

libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet"
}
for lib in libs.values():
    sys.path.append(lib)

Import libraries

In [ ]:
import os
import numpy as np
import scipy as sp
import dask.array as da

from silx.io.dictdump import h5todict
from hypernet import const
from functions import *

Define inputs

In [ ]:
nb_workers = 16
# Plotting
saving = True
showing = True
# Variable to decompose
# variable = {
#     "name": "na",
#     "norm": "$\\tilde{n}_{o}$",
#     "text": r"$n_{o}$ [m$^{-3}$]"
# }
variable = {
    "name": "pa",
    "norm": "$\\tilde{p}_{o}$",
    "text": r"$p_{o}$ [Pa]"
}
# variable = {
#     "name": "T",
#     "norm": "$\\tilde{T}$",
#     "text": r"$T$ [K]"
# }

Define paths

In [ ]:
paths = {
    "figs": f"./figs/{variable['name']}/",
    "tensors": f"./tensors/{variable['name']}/",
    "sols": [
        "./../datagen/runs/samples/data/train/sols.hdf5",
        "./../datagen/runs/samples/data/valid/sols.hdf5"
    ]
}
for k in ("figs", "tensors"):
    os.makedirs(paths[k], exist_ok=True)

Read solutions

In [ ]:
sols = []
for p in paths["sols"]: 
    sols.extend(list(h5todict(p).values()))

Plot solutions

In [ ]:
def get_xy(sol, pscale=False, minmax=False):
    T = sol["y"][:,-1:]
    na = sol["y"][:,:1]
    pa = na * const.UKB * T
    # Input
    x = sol["t"]
    if pscale:
        x = x * pa
    # Output
    if (variable['name'] == "T"):
        y = T
    elif (variable['name'] == "pa"):
        y = pa
    else:
        y = na
    if minmax:
        y = (y - np.amin(y)) / (np.amax(y) - np.amin(y))
    return x.squeeze(), y.squeeze()

> Function of $t$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=False, minmax=True) for sol in sols],
    labels=['$t$ [s]', variable['norm']],
    scales=['log', 'linear'],
    figname=paths["figs"] + f"/data.png",
    save=saving,
    show=showing
)

> Function of $t\cdot p_\text{o}$

In [ ]:
plot_traj(
    xy=[get_xy(sol, pscale=True, minmax=True) for sol in sols],
    labels=[r'$t\cdot p_{o}(t)$ [s Pa]', variable['norm']],
    scales=['log', 'linear'],
    figname=paths["figs"] + f"/data_pscale.png",
    save=saving,
    show=showing
)

Perfoming SVD
> Interpolate solutions onto the same grid

In [ ]:
def interpolate(t, sol):
    x, y = get_xy(sol, pscale=True, minmax=False)
    # Extend domain
    x = np.concatenate([x, x[-1:]*1e2])
    y = np.concatenate([y, y[-1:]])
    # Interpolate
    return sp.interpolate.interp1d(x, y, kind='cubic')(t)

In [ ]:
tp_lim = (-7, 0)
tp = np.logspace(*tp_lim, 1000)

In [ ]:
y = [interpolate(tp, sol) for sol in sols]
y = np.vstack(y).T

> SVD decomposition

In [ ]:
m = np.mean(y, axis=0, keepdims=True)
s = np.std(y, axis=0, keepdims=True)
yn = (y - m) / s

In [ ]:
yd = da.from_array(yn, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=200)
s = s.compute()
u = u.compute()

> Singular values decay

In [ ]:
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)

In [ ]:
# Number of principal components
eps = 1e-5
nb_pc = np.where(cs > 1-eps)[0][0] + 1
nb_pc

In [ ]:
plot_cumenergy(
  cs[:10],
  figname=paths["figs"] + "/cum_en.png",
  save=saving,
  show=showing
)

> Extract modes

In [ ]:
phi = u[:,:nb_pc]
np.savetxt(paths["tensors"] + "/phi.txt", phi, fmt='%.10e')
np.savetxt(paths["tensors"] + "/tp.txt", tp, fmt='%.10e')

In [ ]:
path = paths["figs"] + "/modes/"
os.makedirs(path, exist_ok=True)
for i in range(nb_pc):
    mode_nb = str(i+1)
    plot_2d(
        x=tp,
        y_true=phi[:,i],
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', r"$\phi_{{%s}}$" % mode_nb],
        scales=["log", "linear"],
        filename=path + f"/phi_{mode_nb.zfill(2)}.png",
        save=saving,
        show=showing
    )

> Test reconstruction

In [ ]:
def reconstruct(y):
    m = np.mean(y)
    s = np.std(y)
    yn = (y - m) / s
    w = ols(phi, yn)
    yr = (phi @ w) * s + m
    return yr

In [ ]:
path = paths["figs"] + "/rec/"
os.makedirs(path, exist_ok=True)

In [ ]:
cases = np.random.randint(low=0, high=len(sols), size=10)

In [ ]:
for icase in cases:
    yi = interpolate(tp, sols[icase])
    yr = reconstruct(yi)
    plot_2d(
        x=tp,
        y_true=yi,
        y_pred=yr,
        labels=[r'$t\cdot p_{o}(t)$ [s Pa]', variable['text']],
        scales=["log", "linear"],
        filename=path + f"/{str(icase).zfill(6)}.png",
        save=saving,
        show=showing
    )